# 大语言模型的局限性与幻觉问题

## 1. 为什么大语言模型智能体有时会"胡说八道"？

LLM 的"幻觉"（Hallucination）是内在机制决定的，不是 bug，而是特性的代价：

- **概率生成，而非事实检索**：LLM 本质是预测下一个 token 的概率分布，不是查数据库。它被训练成"说像人话的话"，而不是"说真话"。当训练数据中某个事实出现频率低、或者问题超出了知识边界，模型会"编"一个概率上最合理的回答——但可能完全错误。

- **缺乏真实世界 ground truth 校验**：LLM 没有内置的"事实核查器"。一个人类会说"我不确定，我帮你查一下"，但 LLM 倾向于无论如何都要生成一段流畅的文本。

- **长上下文的注意力稀释**：在第一章的循环中，随着 Thought-Action-Observation 不断在 prompt 中累积，模型对早期信息的注意力会逐渐衰减，导致"忘记"之前的约束条件，开始产生不一致的输出。

**如何缓解？**
- 工具调用（类似 1.3 节的 tavily search），让模型用 API 查真实数据而非凭记忆编造
- RAG（检索增强生成）：先从知识库检索相关文档，再让 LLM 基于文档回答
- 在 prompt 中明确指示"如果不确定，请说不知道"
- 多轮验证：让模型自我检查或引入第二个模型做事实核查

---

## 2. 如果去掉最大循环次数（5 次）限制，会发生什么？

在 1.3 节实现了 `for i in range(5)`，即最多 5 轮 Thought-Action-Observation 循环。如果没有这个限制：

**可能的问题：**

1. **死循环（Infinite Loop）**：如果工具返回错误（如查不到天气）、或模型解析 Action 失败、或模型持续产出格式不正确的输出，循环会永远执行，消耗大量 API 费用且没有任何产出。

2. **目的漂移（Goal Drift）**：随着 prompt history 越来越长，模型可能在多轮对话后忘记最初的目标（"查北京的天气"），转而开始做无关的事情。这在实践中非常常见——agent 跑着跑着就"走神"了。

3. **上下文窗口溢出**：每次循环都往 `prompt_history` 追加 Thought + Action + Observation。如果无限循环，prompt 最终会超过模型的上下文窗口上限（DeepSeek-V4 是 1M tokens），导致最早的轮次被截断，丢失初始任务信息。

4. **资源浪费**：LLM API 是按 token 计费的，一个失控的循环每分钟可能烧掉几十甚至上百元。

**更好的循环设计：**
- 保留最大步数限制，但调高到合理值（如 10-15 次）
- 增加**超时限制**（如 30 秒内必须完成）
- 增加**语义检测**：如果连续 3 轮没有实质性进展（Observation 雷同或为空），自动结束
- 增加**目的再确认**：每隔 3 轮在 prompt 中重新插入原始用户请求

---

## 3. 设计一个"可信度"指标，评估智能体输出的可靠程度

可信度不能是一个简单的"是/否"，需要多维度衡量：

| 维度 | 评估方式 | 举例 |
|------|----------|------|
| **来源可信度** | 输出内容是否来源于工具调用的实际结果？ | 如果天气数据来自 wttr.in API 返回，可信度高；如果是模型"我记得北京通常是晴天"，可信度低 |
| **一致性** | 后续轮次的输出与之前的 Observation 是否一致？ | 如果第2轮说"北京晴天适合户外"，第3轮 Observation 显示"下暴雨"但模型仍推荐户外——矛盾，可信度下降 |
| **完整性** | 是否回答了用户问题的所有部分？ | 用户问"天气+景点推荐"，如果只回答天气没推荐景点：不完整 |
| **可追溯性** | 每一个结论能否追溯到具体的工具调用结果？ | "根据 wttr.in 返回的气温 25°C 和 Tavily 搜索的第3条结果，推荐颐和园" —— 可追溯，可信 |
| **不确定性表达** | 模型是否主动表达了不确定性？ | "建议去颐和园" vs "根据当前信息，颐和园是比较合适的选择，但建议出发前确认门票情况" —— 后者更可信 |

**在代码中的实现思路：**
```python
# 在每个循环中追踪
trust_score = 1.0
if not tool_called:
    trust_score -= 0.3  # 没有调用工具，可能是凭记忆说的
if observation_contradicts_previous:
    trust_score -= 0.5
if '不确定' in llm_output or '可能' in llm_output:
    trust_score += 0.1  # 诚实表达不确定性是加分项

if trust_score < 0.4:
    print('警告：当前回答可信度较低，建议人工复核')
```

> **最终结论**：单纯用大语言模型输出"准不准确"难以判断，更好的思路是结合**工具调用的数据来源 + 逻辑一致性 + 不确定性的诚实表达**来综合打分。这正是第一章反复强调的核心思想：LLM 智能体的能力不在于"知道一切"，而在于"知道何时、如何调用正确的工具来获取真实数据"。